# Config

In [3]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [ ]:
import pandas as pd
import os
from preprocess.preprocess import clean_text
from preprocess.translate import translator, gen_text_for_embedding, final_clean
import time

# 1) Preprocesamiento de los datos


In [31]:
import re
import unicodedata
import pandas as pd

def _short_pat(pat: str, n: int = 50) -> str:
    pat = pat.strip()
    return (pat[:n] + "...") if len(pat) > n else pat

def check_deleted_expressions(texts, return_long: bool = False):
    """
    Detecta (sin eliminar) qué fragmentos coincidirían con los patrones de
    get_expressions_to_delete() en cada texto.

    Parámetros
    ----------
    texts : iterable[str]
        Lista/iterable de textos.
    return_long : bool, opcional (default=False)
        Si True, además del pivote ancho devuelve un DataFrame largo con una fila por match.

    Devuelve
    --------
    df_wide : pd.DataFrame
        Filas = text_idx, Columnas = patrón (string acortado), Valores = coincidencias concatenadas.
    df_long (opcional) : pd.DataFrame
        Columnas: text_idx, pattern_id, pattern, start, end, match.
    """
    # Preparar patrones
    patterns = get_expressions_to_delete()
    pat_meta = [
        {"id": f"P{i+1}", "obj": p, "name": _short_pat(p.pattern)}
        for i, p in enumerate(patterns)
    ]

    # Recolectar matches
    records = []
    for i, raw_text in enumerate(texts):
        # Normalización básica (sin eliminar nada por regex)
        text = raw_text if isinstance(raw_text, str) else ""
        text = unicodedata.normalize("NFKC", text)
        text = re.sub(r'[\u00A0\u1680\u180E\u2000-\u200F\u202F\u205F\u3000\uFEFF]', ' ', text)
        text = re.sub(r'\_x000D_', ' ', text)

        for meta in pat_meta:
            pat = meta["obj"]
            for m in pat.finditer(text):
                match_txt = m.group(0)
                records.append({
                    "text_idx": i,
                    "pattern_id": meta["id"],
                    "pattern": meta["name"],
                    "start": m.start(),
                    "end": m.end(),
                    "match": match_txt.strip()
                })

    # Si no hubo coincidencias, devolver DataFrame vacío consistente
    if not records:
        df_wide = pd.DataFrame(columns=["text_idx"] + [m["name"] for m in pat_meta])
        df_wide.set_index("text_idx", inplace=True)
        return (df_wide, pd.DataFrame(columns=["text_idx", "pattern_id", "pattern", "start", "end", "match"])) if return_long else df_wide

    # DF largo
    df_long = pd.DataFrame(records)

    # Agregar múltiples matches por (text_idx, pattern) y concatenar sin duplicados preservando orden
    agg = (
        df_long.groupby(["text_idx", "pattern"], as_index=False)["match"]
        .apply(lambda s: " | ".join(dict.fromkeys([x for x in s if x])))
    )

    # Pivot a formato ancho
    df_wide = agg.pivot(index="text_idx", columns="pattern", values="match").fillna("")

    # Asegurar columnas para todos los patrones (aunque queden vacías)
    all_cols = [m["name"] for m in pat_meta]
    for c in all_cols:
        if c not in df_wide.columns:
            df_wide[c] = ""
    df_wide = df_wide.reindex(columns=all_cols).sort_index()

    if return_long:
        return df_wide, df_long.sort_values(["text_idx", "start"])
    return df_wide


In [ ]:
# 1) Cargar datos
path = "/tmp/data"
path_analytics = "/tmp/analytics"
filePATH = os.path.join(path, "data_concatenada.xlsx")
df = pd.read_excel(filePATH,
                   usecols=["Código VRID", "Título", "Resumen", "Keywords", "Interdisciplinario", "Transdisciplinario", "Facultad del Proyecto",
                            "Depto Persona"]) \
       .fillna("")

# 2) Guardar qué secuencias de palabras del resumen serán eliminadas al aplicar get_expressions_to_delete()
list_texts = df["Resumen"].to_list()
df_deleted = check_deleted_expressions(list_texts)
savepath=os.path.join(path_analytics, "deleted_re.xlsx")
df_deleted.to_excel(savepath, index=False)

# 3) Preprocesar los datos
#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad",
    "Depto Persona": "Depto_Persona_trad",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)
savepath=os.path.join(path, "data_clean.xlsx")
df.to_excel(savepath, index=False)

# 2) Traducción del texto

## Translator

In [10]:
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
#####Estoy trabajando en mejorar esta parte para que sea más rápida con paralelización por batches
start = time.time()
for src, dst in cols.items():
    df[dst] = trans.translate_parallel(df[src].to_list(), batch_size=8)
end = time.time()


#3.Guardado de resultados
savepath=os.path.join(path, "data_translated.xlsx")
df.to_excel(savepath, index=False)

print(f"Tiempo total de traducción: {end - start:.2f} segundos")

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Usando dispositivo: cuda


model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Traduciendo: 100%|██████████| 119/119 [00:27<00:00,  4.40batch/s]


Tiempo total de traducción: 769.17 segundos


In [16]:
#3. Selección de columnas que se utilizarán en clasificador y concatenación
# Última limpieza antes de generar concatenación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad", "Facultad_del_Proyecto_trad", "Depto_Persona_trad"]
for col in cols:
    df[col] = df[col].apply(final_clean)

#  Selección de columnas para embedding.
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
df = gen_text_for_embedding(df, cols)

# Guardado de resultados
savepath=os.path.join(path, "data_translated_concat.xlsx")
df.to_excel(savepath, index=False)
savepath=os.path.join(path, "data_translated_concat.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

df.head()

,Código VRID,Interdisciplinario,Transdisciplinario,Título,Keywords,Resumen,Facultad del Proyecto,Depto Persona,Titulo_trad,Resumen_trad,keywords_trad,Facultad_del_Proyecto_trad,Depto_Persona_trad,text_for_embedding_translated
0,217.173.049-1.0,SI,,PATRONES DE CRIANZA Y SOCIALIZACIÓN DE GÉNERO ...,,OBJETIVOS GENERALES: _x000D_\nDESCRIBIR LOS PR...,FACULTAD DE CIENCIAS SOCIALES,"DEPARTAMENTO DE ECONOMÍA, SIN INFORMACIÓN, ESC...",patterns of gender upbringing and socializatio...,general objectives: to describe the processes ...,,faculty of social sciences,"department of economics, without information, ...",patterns of gender upbringing and socializatio...
1,218.201.002-1.0,SI,,ADAPTACIÓN CULTURAL Y VALIDACIÓN DE LA ESCALA ...,"ESTILO DE VIDA, ADOLESCENTES _x000D_\n",PARA EVALUAR LOS COMPORTAMIENTOS RELACIONADOS ...,FACULTAD DE ENFERMERÍA,"DEPARTAMENTO DE CIENCIA ANIMAL, DEPARTAMENTO D...",cultural adaptation and validation of the life...,in order to evaluate the behaviors related to ...,"lifestyle, teens",faculty of nursing,"department of animal science, department of pl...",cultural adaptation and validation of the life...
2,218.102.031-1.0IN,NO,,PROMOVIENDO LA REFLEXIÓN EN ESTUDIANTES DE PRE...,,EL PRESENTE PROYECTO INVOLUCRA LA REALIZACIÓN ...,FACULTAD DE ODONTOLOGÍA,DEPARTAMENTO DE ASTRONOMÍA,promoting reflection in preclinical dental stu...,the present project involves the realization o...,,faculty of dentistry,department of astronomy,promoting reflection in preclinical dental stu...
3,218.163.016-INI,INDEFINIDO,,MOTIVACIÓN Y HABILIDADES SOCIALES EN ADOLESCENTES,,EL ESTUDIO DE LA MOTIVACIÓN TIENE DIFERENTES A...,FACULTAD DE EDUCACIÓN,"DEPARTAMENTO DE CIENCIAS DE LA EDUCACIÓN, DEPT...",motivation and social skills in adolescents,the study of the motivation has different side...,,faculty of education,"department of education sciences, department o...",motivation and social skills in adolescents t...
4,219.091.052-INI,NO,,TIME EFFECTS ON THE LIQUEFACTION RESPONSE OF G...,,SECONDARY CONSOLIDATION AND AGEING ARE TWO OFT...,FACULTAD DE INGENIERÍA,DEPTO. TEORÍA POLITICA Y FUND.DE LA EDUC.,time effects on the liquefaction response of g...,secondary consolidation and ageing are two oft...,,faculty of engineering,department of political and fund theory of edu...,time effects on the liquefaction response of g...
